# IEEE-CIS: How Much of Your Random-CV AUC Is Leakage?

IEEE-CIS Fraud Detection has a famous trap. Random k-fold CV reports an AUC well above what the
leaderboard gives back. The test set is *later in time* than train, and a random split lets the
model see the same customers on both sides of the fold.

This notebook measures the trap instead of just naming it, on the same model and features:

1. random 5-fold vs a time-ordered split vs forward-chaining by month;
2. how many validation frauds belong to a customer the model already saw in training, under each split;
3. the AUC on *seen* vs *unseen* customers, which is where the gap comes from.

Everything runs on the competition files: ten LightGBM fits, about 2.5 hours on Kaggle CPU.

In [ ]:
import gc, glob, os, time
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

T0 = time.time()
hits = sorted(glob.glob("/kaggle/input/**/train_transaction.csv", recursive=True))
assert hits, f"competition data not attached: {os.listdir('/kaggle/input')}"
D = os.path.dirname(hits[0])
tr = pd.read_csv(f"{D}/train_transaction.csv")
idn = pd.read_csv(f"{D}/train_identity.csv")
tr = tr.merge(idn, on="TransactionID", how="left")
del idn; gc.collect()
tr = tr.sort_values("TransactionDT").reset_index(drop=True)
y = tr["isFraud"].to_numpy()
print(tr.shape, f"fraud rate {y.mean():.4f}", f"({time.time() - T0:.0f}s)")

## Features and a customer key

The features are deliberately plain: every column as given, with categoricals label-encoded. No
engineered aggregates, so the only difference between runs is the split.

The customer key is the well-known reconstruction from this competition: card1 + addr1 + the
account-opening day, where D1 counts days since opening, so `day - D1` stays constant for one
account. It is approximate, but good enough to count who was seen before.

In [ ]:
for c in tr.columns:
    if tr[c].dtype == object:
        tr[c] = tr[c].astype("category").cat.codes.astype("int16")
    elif tr[c].dtype == "float64":
        tr[c] = tr[c].astype("float32")
day = (tr["TransactionDT"] // 86400).astype(int)
uid = (tr["card1"].astype(str) + "_" + tr["addr1"].astype(str) + "_" + (day - tr["D1"]).astype(str)).to_numpy()
month = (tr["TransactionDT"] // (30 * 86400)).astype(int).to_numpy()
X = tr.drop(columns=["isFraud", "TransactionID", "TransactionDT"])
print(X.shape[1], "features |", len(np.unique(uid)), "customer keys | months", np.bincount(month - month.min()))

In [ ]:
PARAMS = dict(n_estimators=600, learning_rate=0.05, num_leaves=256, max_depth=-1, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.4, min_child_samples=100, reg_lambda=1.0, verbose=-1, n_jobs=-1, random_state=42)

def fit_score(trn, val):
    m = lgb.LGBMClassifier(**PARAMS)
    m.fit(X.iloc[trn], y[trn])
    p = m.predict_proba(X.iloc[val])[:, 1]
    seen = np.isin(uid[val], np.unique(uid[trn]))
    fr = y[val] == 1
    return p, seen, {"AUC": roc_auc_score(y[val], p), "val frauds from seen customers": seen[fr].mean()}

results, per_row = {}, {}

## 1. Three ways to validate the same model

In [ ]:
# (a) random stratified 5-fold, the default everyone starts with
oof, seen_all = np.zeros(len(X)), np.zeros(len(X), bool)
rows = []
for k, (trn, val) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y)):
    p, seen, r = fit_score(trn, val)
    oof[val], seen_all[val] = p, seen
    rows.append(r)
    print(f"random fold {k}: AUC {r['AUC']:.4f}  ({time.time() - T0:.0f}s)")
results["random 5-fold"] = {"AUC": roc_auc_score(y, oof), "val frauds from seen customers": seen_all[y == 1].mean()}
per_row["random 5-fold"] = (oof, seen_all, np.ones(len(X), bool))

In [ ]:
# (b) time-ordered: train on the first 80% of TransactionDT, validate on the last 20% (test is later in time)
cut = int(len(X) * 0.8)
trn, val = np.arange(cut), np.arange(cut, len(X))
p, seen, r = fit_score(trn, val)
results["time split 80/20"] = r
mask = np.zeros(len(X), bool); mask[val] = True
o = np.zeros(len(X)); o[val] = p
s = np.zeros(len(X), bool); s[val] = seen
per_row["time split 80/20"] = (o, s, mask)
print(r, f"({time.time() - T0:.0f}s)")

In [ ]:
# (c) forward chaining: train on months < m, validate on month m
ms = np.unique(month)
fc = []
for m_ in ms[2:]:
    trn, val = np.flatnonzero(month < m_), np.flatnonzero(month == m_)
    _, _, r = fit_score(trn, val)
    fc.append(r)
    print(f"month {m_}: AUC {r['AUC']:.4f}  seen-customer frauds {r['val frauds from seen customers']:.2f}")
results["forward chaining (mean)"] = {k: float(np.mean([r[k] for r in fc])) for k in fc[0]}
pd.DataFrame(results).T.round(4)

## 2. Where the gap comes from: customers the model has already met

Split every evaluation into validation rows whose customer key also appears in that fold's
training data (*seen*) and rows whose key does not (*unseen*).

In [ ]:
rows = []
for name, (p, seen, mask) in per_row.items():
    for label, sel in (("seen", mask & seen), ("unseen", mask & ~seen)):
        if y[sel].min() != y[sel].max():
            rows.append({"split": name, "customers": label, "rows": int(sel.sum()), "fraud rate": y[sel].mean(),
                         "AUC": roc_auc_score(y[sel], p[sel])})
breakdown = pd.DataFrame(rows).round(4)
print(breakdown.to_string(index=False))

In [ ]:
b = breakdown.set_index(["split", "customers"]).AUC
fr = {n: float(seen[mask & (y == 1)].mean()) for n, (_, seen, mask) in per_row.items()}
print(f"validation frauds from seen customers: random {fr['random 5-fold']:.0%} | time split {fr['time split 80/20']:.0%}")
print(f"gap random vs time split:           {results['random 5-fold']['AUC'] - results['time split 80/20']['AUC']:+.4f}")
print(f"seen vs unseen, random split:       {b['random 5-fold', 'seen'] - b['random 5-fold', 'unseen']:+.4f}  (customer overlap)")
print(f"random vs time, unseen customers:   {b['random 5-fold', 'unseen'] - b['time split 80/20', 'unseen']:+.4f}  (time drift)")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
res = pd.DataFrame(results).T
ax[0].barh(res.index, res["AUC"], color=["#c4553a", "#3a7dc4", "#3a7dc4"])
ax[0].set_xlim(res["AUC"].min() - 0.03, 1); ax[0].set_title("validation AUC by split")
b = breakdown.pivot(index="split", columns="customers", values="AUC")
b.plot.barh(ax=ax[1], color={"seen": "#c4553a", "unseen": "#3a7dc4"})
ax[1].set_xlim(0.8, 1); ax[1].set_title("AUC on seen vs unseen customers")
plt.tight_layout(); plt.show()
print(f"total runtime {time.time() - T0:.0f}s")

## Takeaways

- **Random k-fold answers a question the leaderboard never asks.** Under a random split most
  validation frauds belong to customers whose other transactions are in training, and the model scores
  those customers far better than new ones. That is half of the gap.
- **The other half is time.** Even on customers it has never seen, random CV stays optimistic, because
  its folds mix every month. Fraud patterns drift, and a random fold never has to predict the future.
- **The time split is the honest number.** It matches how test relates to train (later in time)
  and puts far fewer seen customers in validation.
- **Choose features on the split you trust.** A feature that helps only under random CV is usually
  a customer identifier in disguise. Check what it does on *unseen* customers before keeping it.
- Seen-customer effects are real at test time too (test shares customers with train). The point is
  not to eliminate them, but not to let them decide which model you ship.

*Part of a series on why validation and leaderboards disagree:
[House Prices](https://www.kaggle.com/code/senanuretin/house-prices-my-cv-said-0-1056-the-lb-disagreed) ·
[CASMI 2026](https://www.kaggle.com/code/senanuretin/casmi26-a-bigger-candidate-pool-lowered-my-score).*